# LSI-lite (Colab) — Figure / Mark-Making / Landscape (+ Color and Tonal Telemetry)

Small, profiled composition gate for quick QA of images.  
LSI-lite measures how an image behaves under compositional structure using three primitives: Δx (off-center gravity), rᵥ (void ratio), ρᵣ (rupture/mark energy) and tells you if it sits within intended bands for its class. It’s built to study stability, not to crown winners.
Balance (Δx): How far the visual center is from the geometric center
Density (rᵥ): The ratio of empty space to filled space
Detail (ρᵣ): The amount of edge energy and texture density in key areas
It combines these measurements into a 0-100 score for how an image lines up or "passes" basic structural compositional criteria. It helps distinguish delta in AI and human default.

Quick Start

1. Run Setup & Config
- This installs requirements and sets defaults. (No need to tweak pins unless you want custom paths.)
2. (Optional) Enable Color/Tonal telemetry (enabled)
- By default, color checks are on with COLOR_SPACE="LAB".
- You can turn them off by setting INCLUDE_COLOR=False.
3. Upload your images
4. Choose Profile mode, use auto for ease or select
4. Run Scoring
- The notebook will compute Δx, rᵥ, ρᵣ (and color/tonal if enabled).
- Tables show per-image scores and pass/fail badges.
6. Plots visualize centroid drift, void ratio, and stroke energy.
- Audit badges (color/tonal) only appear if thresholds trip.

**Profiles:** `Figure_Default`, `MarkMaking_Expressive`, `Landscape` (band guards + weights).  

**Color mask details (telemetry):** LAB k-means (k=3); largest cluster → background; foreground = union of the others + morphology. Optional external uint8 mask (0/255) can override (mask_mode='external'). If color fails/unavailable, we log fallback_gray. Tonal metrics always run on the L channel using the same ROI as Δx (if active).

**Color & Tonal telemetry (advisory only)::** color — rv_color_mask, dx_color_L, delta_rv, delta_dx, mask_mode (gray|color|external), color_space="LAB", color_status (ok|fallback_gray|fail_convert).
tonal — S_L (P95−P5 on L), eta_L (Otsu separability), beta_L (bright mass ≥0.80), tonal_status.
Badges: color_audit_badge, tonal_audit_badge (auto-combined if both fire). ROI-aligned with Δx; Landscape uses auto reflection ROI.

S_L = tonal span on L (P95 − P5).
eta_L = Otsu separability (0–1).
beta_L = bright-mass fraction (L ≥ 0.80).
delta_dx = dx_color_L − Δx, delta_rv = rv_color_mask − rᵥ.

> This is **not** recognition or a “style police” or a judgement on "aesthetics." It’s a tiny, defensible ruler over three compositional 101 primitives with color and tonal - aware telemetry to clarify why a frame passes or fails. Think of it as a quick ruler for balance, void, and stroke coherence. LSI-lite as a complementary metric in the generative AI evaluation ecosystem.


In [ ]:
# @title Setup & Config (resilient, pinned when requested)
PIN_DEPS = False  # @param {type:"boolean"}

PINS = {
    "opencv-python-headless": "4.10.0.84",
    "numpy": "1.26.4",
    "pandas": "2.0.3",
    "matplotlib": "3.7.5",
    "Pillow": "10.4.0",
}

def maybe_pin_deps(pin=PIN_DEPS):
    """Install pinned versions only if requested; otherwise use Colab defaults."""
    if not pin:
        print("Pinned deps disabled. Using environment defaults.")
        return
    import sys, subprocess
    pkgs = [f"{k}=={v}" for k, v in PINS.items()]
    cmd = [sys.executable, "-m", "pip", "install",
           "--prefer-binary", "--only-if-needed", "--upgrade-strategy", "only-if-needed"] + pkgs
    print("Installing pinned:", " ".join(pkgs))
    try:
        subprocess.check_call(cmd)
        print("Pin install finished.")
    except Exception as e:
        print("Pin install failed; continuing with existing env:", e)

maybe_pin_deps()

# now the normal imports
import os, glob, math, io, json, shutil, base64
import random
import numpy as np, cv2, pandas as pd
np.random.seed(0)
random.seed(0)
cv2.setRNGSeed(0)
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

os.makedirs("/content/images", exist_ok=True)

CONFIG = {
    "preprocessing": {"longest_side": 1536, "morph_kernel": 5},
    "accept": {"gate_100": 55.0},
    "sigma_scale": 0.35,
}

PROFILES = {
    "Figure_Default": {
        "weights": {"dx": 0.45, "rv": 0.35, "rho": 0.20},
        "bands": {
            "dx":  {"guard": [0.05, 0.85]},
            "rv":  {"guard": [0.10, 0.90]},
            "rho": {"guard": [0.10, 0.80]},
        },
        "rho_mask": {"type": "subject_halo", "halo_frac": 0.08},
        "dx_roi": None,
    },
    "MarkMaking_Expressive": {
        "weights": {"dx": 0.40, "rv": 0.30, "rho": 0.30},
        "bands": {
            "dx":  {"guard": [0.02, 0.90]},
            "rv":  {"guard": [0.10, 0.90]},
            "rho": {"guard": [0.06, 0.70]},
        },
        "rho_mask": {"type": "subject_halo", "halo_frac": 0.12},  # ← keep only this one
        "dx_roi": None,
    },
    "Landscape": {
        "weights": {"dx": 0.35, "rv": 0.40, "rho": 0.25},
        "bands": {
            "dx":  {"guard": [0.05, 0.95]},
            "rv":  {"guard": [0.10, 0.90]},
            "rho": {"guard": [0.08, 0.80]},
        },
        "rho_mask": {"type": "full"},
        "dx_roi": "auto",   # ← turn on reflection-aware Δx crop (or set to None to disable)
    },
}
print("Ready. Profiles:", list(PROFILES.keys()))
# --- Color telemetry toggles (does NOT affect gating) ---
INCLUDE_COLOR = True      # flip to False to disable all color telemetry
COLOR_SPACE   = "LAB"     # LAB (OpenCV) or OKLab if you add it later

COLOR_TELEMETRY_FIELDS = [
    # color telemetry
    "rv_color_mask","dx_color_L","delta_rv","delta_dx",
    "mask_mode","color_space","color_status","color_audit_badge",
    # tonal telemetry
    "S_L","eta_L","beta_L","tonal_status","tonal_audit_badge",
]

In [ ]:
# @title Helpers (robust loader, safe largest, morphology, masks, auto-profile)

def load_image_robust(path_or_bytes):
    """Open as RGB with EXIF orientation; return grayscale uint8 and original RGB HxWx3."""
    if isinstance(path_or_bytes, (bytes, bytearray)):
        img = Image.open(io.BytesIO(path_or_bytes))
    else:
        img = Image.open(path_or_bytes)
    img = ImageOps.exif_transpose(img).convert("RGB")
    rgb = np.asarray(img)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    return gray, rgb

def resize_longest(gray, longest):
    h, w = gray.shape[:2]
    s = longest / max(h, w)
    if s < 1.0:
        gray = cv2.resize(gray, (int(w*s), int(h*s)), interpolation=cv2.INTER_AREA)
    return gray

def safe_largest(bin_u8: np.ndarray):
    cnts, _ = cv2.findContours(bin_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    a = max(cnts, key=cv2.contourArea)
    mask = np.zeros_like(bin_u8, dtype=np.uint8)
    cv2.drawContours(mask, [a], -1, 255, -1)
    return mask

def _morph(mask, k):
    kernel = np.ones((k, k), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=1)
    return mask

def foreground_mask(gray: np.ndarray, cfg: dict) -> np.ndarray:
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, bin_ = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Prefer “dark subject on light paper”, then try normal, then degenerate fallback
    mask = safe_largest((255 - bin_).astype(np.uint8))
    if mask is None:
        mask = safe_largest(bin_.astype(np.uint8))
    if mask is None:
        mask = (bin_ > 0).astype(np.uint8) * 255

    # Light clean-up
    k = cfg["preprocessing"]["morph_kernel"]
    kernel = np.ones((k, k), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, 1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, 1)
    return mask

def subject_halo_mask(fg_mask, halo_frac=0.10):
    h, w = fg_mask.shape[:2]
    r = max(1, int(halo_frac * min(h, w)))
    kernel = np.ones((r, r), np.uint8)
    halo = cv2.dilate(fg_mask, kernel, iterations=1)
    return halo

def auto_profile(dx, rv):
    # very small dx + high rv => Landscape; mid dx / mid rv => Figure; else MarkMaking
    if rv >= 0.35 and dx <= 0.25:
        return "Landscape"
    if rv <= 0.55 and dx >= 0.15:
        return "Figure_Default"
    return "MarkMaking_Expressive"

def band_center_and_span(profile: str, band: str):
    """Return (center, half-span) from the guard band for a given profile/band."""
    g = PROFILES[profile]["bands"][band]["guard"]  # [lo, hi]
    c = 0.5 * (g[0] + g[1])
    s = max(1e-6, 0.5 * (g[1] - g[0]))
    return c, s

def ensure_gray_array(x):
    """Accept path | ndarray | tuple and return a 2-D uint8 grayscale array."""
    # If it's a path/bytes, load it
    if isinstance(x, (str, bytes, bytearray)):
        g = load_image_robust(x)
    else:
        g = x

    # Some code paths hand us (gray, extra). Take the first item.
    if isinstance(g, tuple):
        g = g[0]

    g = np.asarray(g)
    # If RGB/BGR, convert to gray
    if g.ndim == 3 and g.shape[2] in (3, 4):
        # assume RGB because load_image_robust returns RGB
        g = cv2.cvtColor(g, cv2.COLOR_RGB2GRAY)
    # Ensure uint8
    if g.dtype != np.uint8:
        g = np.clip(g, 0, 255).astype(np.uint8)
    return g

def auto_dx_roi(gray_or_tuple,
                *,
                top_frac: float = 0.60,
                reflect_thr: float = 0.35,     # was 0.50; looser to catch real lakes
                waterish_ratio: float = 1.15,  # was 1.30; horizontal > vertical
                require_symmetry: bool = False,
                sym_eps: float = 0.25):
    """
    Auto-crop ROI for Δx when a water reflection is present (Landscape).
    Accepts path/ndarray/(gray,rgb) tuple. Returns dict {"type":"top_frac","top":<f>}
    or None when no crop is recommended.

    Heuristic:
      1) Reflection similarity between top half and flipped bottom half (edges).
      2) "Water-ish" bottom: horizontal > vertical gradients.
      3) (optional) Similar foreground fill in top/bottom halves.
    """
    # 1) Get a 2-D uint8 grayscale image, regardless of input type
    g = ensure_gray_array(gray_or_tuple)   # your helper: path/tuple-safe → 2-D uint8
    if g is None or g.ndim != 2:
        return None
    H, W = g.shape
    if H < 4 or W < 4:
        return None

    h2  = H // 2
    top = g[:h2, :]
    bot = g[h2:, :]

    # 2) Edge maps for similarity probe
    e_top = cv2.Canny(top, 50, 100).astype(np.float32)
    e_bot = cv2.Canny(bot, 50, 100).astype(np.float32)
    e_bot_flip = np.flipud(e_bot)

    # Cosine similarity between the two halves’ edge maps
    num = (e_top * e_bot_flip).sum()
    den = float(np.linalg.norm(e_top) * np.linalg.norm(e_bot_flip) + 1e-6)
    reflect_sim = num / den

    # 3) "Water-ish" check: horizontal structure should dominate in the bottom half
    gx = np.abs(cv2.Sobel(bot, cv2.CV_32F, 1, 0, ksize=3)).mean()
    gy = np.abs(cv2.Sobel(bot, cv2.CV_32F, 0, 1, ksize=3)).mean()
    waterish = gx > (waterish_ratio * gy)

    # 4) Optional symmetric foreground fill (uses your existing CONFIG + mask helper)
    ok_symmetry = True
    if require_symmetry:
        mask = foreground_mask(g, CONFIG)       # relies on global CONFIG (as in your notebook)
        fill_top = (mask[:h2, :] > 0).mean()
        fill_bot = (mask[h2:, :] > 0).mean()
        ok_symmetry = abs(fill_top - fill_bot) < sym_eps

    # 5) Decision
    if (reflect_sim > reflect_thr) and waterish and ok_symmetry:
        return {"type": "top_frac", "top": float(top_frac)}
    return None

In [ ]:
# Color helpers (telemetry only)

import cv2, numpy as np

def _largest_component_uint8(mask_u8: np.ndarray) -> np.ndarray:
    """Return only the largest connected component from a binary uint8 mask."""
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    if num <= 1:
        return mask_u8
    # skip label 0 (background); take argmax by area
    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_idx = 1 + np.argmax(areas)
    keep = (labels == largest_idx).astype(np.uint8) * 255
    return keep

def color_mask_via_lab_kmeans(rgb_u8: np.ndarray, k: int = 3, downsample: int = 2, morph_kernel: int = 5) -> np.ndarray:
    """
    Build a foreground mask from color using LAB k-means (k=3).
    Heuristic: largest cluster = background; foreground = union of the rest.
    """
    if rgb_u8 is None:
        raise ValueError("color_mask_via_lab_kmeans: rgb_u8 is None")
    h, w = rgb_u8.shape[:2]
    # Convert to LAB
    lab = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2LAB)
    # Downsample for faster k-means
    if downsample > 1:
        lab_small = cv2.resize(lab, (w // downsample, h // downsample), interpolation=cv2.INTER_AREA)
    else:
        lab_small = lab
    Z = lab_small.reshape((-1, 3)).astype(np.float32)

    # K-means
    cv2.setRNGSeed(0)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    attempts = 3
    flags = cv2.KMEANS_PP_CENTERS
    compactness, labels, centers = cv2.kmeans(Z, k, None, criteria, attempts, flags)

    labels = labels.reshape(lab_small.shape[:2])
    # Majority cluster → background
    bg_label = np.bincount(labels.flatten()).argmax()
    fg_small = (labels != bg_label).astype(np.uint8) * 255

    # Upsample to original size if needed
    if downsample > 1:
        fg = cv2.resize(fg_small, (w, h), interpolation=cv2.INTER_NEAREST)
    else:
        fg = fg_small

    # Morphology cleanup
    K = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_kernel, morph_kernel))
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, K)
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, K)

    # Keep largest component only
    fg = _largest_component_uint8(fg)
    return fg

def compute_rv_from_mask(mask_u8: np.ndarray) -> float:
    """rv = 1 - fill, where fill is fraction of foreground (mask>0)."""
    total = mask_u8.size
    fill = float(np.count_nonzero(mask_u8)) / float(total)
    return max(0.0, min(1.0, 1.0 - fill))

def compute_dx_color(gray_L_u8: np.ndarray, mask_u8: np.ndarray) -> float:
    """
    Δx on L-channel: edge-first centroid restricted by color-foreground mask; fallback to mask centroid.
    Returns normalized horizontal distance [0..1] or np.nan if nothing usable.
    """
    H, W = gray_L_u8.shape[:2]
    # Edge-first within mask
    edges = cv2.Canny(gray_L_u8, 50, 100)
    edges = cv2.bitwise_and(edges, edges, mask=mask_u8)
    if np.count_nonzero(edges) > 0:
        m = cv2.moments(edges)
        if m["m00"] != 0:
            cx = m["m10"] / m["m00"]
            dx = abs(cx - (W / 2.0)) / (W / 2.0)
            return float(max(0.0, min(1.0, dx)))
    # Fallback: mask centroid
    if np.count_nonzero(mask_u8) > 0:
        m = cv2.moments(mask_u8)
        if m["m00"] != 0:
            cx = m["m10"] / m["m00"]
            dx = abs(cx - (W / 2.0)) / (W / 2.0)
            return float(max(0.0, min(1.0, dx)))
    return float("nan")

def color_telemetry(gray_u8: np.ndarray, rgb_u8: np.ndarray, morph_kernel: int = 5):
    """
    Compute (rv_color, dx_color, mask_mode, color_status).
    Never raises; on any failure, returns (None, None, 'none', 'error').
    """
    try:
        if rgb_u8 is None:
            return None, None, "none", "error"
        # Use L from LAB for Δx_color
        lab = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2LAB)
        L = lab[:, :, 0]
        # Build color-foreground mask and compute metrics
        fg_color = color_mask_via_lab_kmeans(rgb_u8, k=3, morph_kernel=morph_kernel)
        rv_c = compute_rv_from_mask(fg_color)
        dx_c = compute_dx_color(L, fg_color)
        return rv_c, dx_c, "color", "ok"
    except Exception:
        return None, None, "none", "error"

# --- Color/Tonal audit helpers (advisory only) ---

def near_guard_rv(rv, pf, tol=0.02):
    lo, hi = PROFILES[pf]["bands"]["rv"]["guard"]
    return (abs(rv - lo) <= tol) or (abs(rv - hi) <= tol)

def apply_roi_to_L(L_u8, dx_roi):
    """Mirror Δx ROI for luminance reads."""
    if isinstance(dx_roi, dict) and dx_roi.get("type") == "top_frac":
        H = L_u8.shape[0]
        return L_u8[: max(1, int(H * float(dx_roi["top"]))), :]
    return L_u8

def tonal_stats_L(L_u8):
    """Return (S_L, eta_L, beta_L) on 8-bit L in [0..255]."""
    import numpy as np, cv2
    # Span (P95 - P5) scaled to [0..1]
    p5, p95 = np.percentile(L_u8, 5), np.percentile(L_u8, 95)
    S_L = float((p95 - p5) / 255.0)

    # Otsu separability η in [0,1]
    hist = cv2.calcHist([L_u8], [0], None, [256], [0, 256]).ravel()
    p = hist / max(1, hist.sum())
    bins = np.arange(256)
    mu_T = (p * bins).sum()
    sigma_T2 = (p * (bins - mu_T) ** 2).sum() + 1e-9
    w0 = np.cumsum(p); w1 = 1.0 - w0
    mu0 = np.cumsum(p * bins) / np.maximum(w0, 1e-9)
    mu1 = (mu_T - np.cumsum(p * bins)) / np.maximum(w1, 1e-9)
    sigma_B2 = (w0 * (mu0 - mu_T) ** 2 + w1 * (mu1 - mu_T) ** 2).max()
    eta_L = float(np.clip(sigma_B2 / sigma_T2, 0.0, 1.0))

    # Bright mass β_L (≥ 0.80)
    beta_L = float((L_u8 >= int(0.80 * 255)).mean())
    return S_L, eta_L, beta_L

In [ ]:

# @title Core primitives
import numpy as np

def centroid_delta_x(gray, cfg, dx_roi=None, mode="hybrid_masked"):
    g = ensure_gray_array(gray)

    # Optional Landscape crop (top fraction)
    if isinstance(dx_roi, dict) and dx_roi.get("type") == "top_frac":
        top = float(dx_roi.get("top", 0.60))
        H = g.shape[0]
        g = g[: max(1, int(H * top)), :]

    g = resize_longest(g, cfg["preprocessing"]["longest_side"])
    g = cv2.bilateralFilter(g, d=5, sigmaColor=25, sigmaSpace=25)

    # Foreground mask
    fg = foreground_mask(g, cfg)
    fg_u8 = (fg > 0).astype(np.uint8) * 255

    # Edge map
    e = cv2.Canny(g, 50, 100).astype(np.uint8)

    # --- masked-edges first (hybrid), then fallback to mask centroid ---
    m = None
    if mode in ("edge", "edge_masked", "hybrid_masked"):
        e_use = e if mode == "edge" else cv2.bitwise_and(e, fg_u8)
        m = cv2.moments(e_use)

    if m is None or m["m00"] <= 1e-6:
        m = cv2.moments(fg_u8)
        if m["m00"] <= 1e-6:
            return float("nan")  # ← was 0.0

    cx = m["m10"] / (m["m00"] + 1e-6)
    W  = g.shape[1]
    dx = abs(cx - (W / 2.0)) / (W / 2.0)
    return float(np.clip(dx, 0.0, 1.0))

def void_ratio(gray, cfg):
    gray = ensure_gray_array(gray)
    gray = resize_longest(gray, cfg["preprocessing"]["longest_side"])
    mask = foreground_mask(gray, cfg)
    fill = (mask > 0).mean()
    rv = float(np.clip(1.0 - fill, 0.0, 1.0))  # more background = higher void
    return rv

def rho_r(gray, cfg, rho_mask_cfg):
    gray = ensure_gray_array(gray)
    gray = resize_longest(gray, cfg["preprocessing"]["longest_side"])
    gray = cv2.bilateralFilter(gray, d=5, sigmaColor=25, sigmaSpace=25)
    fg = foreground_mask(gray, cfg)
    if rho_mask_cfg.get("type") == "subject_halo":
        m = subject_halo_mask(fg, halo_frac=rho_mask_cfg.get("halo_frac", 0.10))
    else:
        m = np.ones_like(fg, dtype=np.uint8)*255
    lap = cv2.Laplacian(gray, cv2.CV_32F, ksize=3)
    energy = np.abs(lap) / 255.0
    sel = energy[m > 0]
    if sel.size == 0:
        return 0.0
    val = float(sel.mean())
    # gentle scale to [0,1]
    return float(np.clip(val * CONFIG.get("rho_scale", 4.0), 0.0, 1.0))

def measure_primitives(path_or_gray, cfg, rho_mask_cfg, dx_roi=None):
    gray = load_image_robust(path_or_gray) if isinstance(path_or_gray, (str, bytes)) else path_or_gray
    dx  = centroid_delta_x(gray, cfg, dx_roi=dx_roi)
    rv  = void_ratio(gray, cfg)
    rho = rho_r(gray, cfg, rho_mask_cfg)
    return {"dx": dx, "rv": rv, "rho": rho}


In [ ]:

# @title Scoring kernel

def band_flag(val, guard):
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return "RED"
    lo, hi = guard
    return "OK" if (lo <= val <= hi) else "RED"

def gaussian_score(val, guard):
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return 0.0
    lo, hi = guard
    c = 0.5 * (lo + hi)
    sigma = max(1e-6, CONFIG.get("sigma_scale", 0.25) * (hi - lo))
    return float(np.clip(math.exp(-0.5 * ((val - c) / sigma) ** 2), 0.0, 1.0))

def score_with_profile(prims, profile_cfg, cfg):
    bands = profile_cfg["bands"]
    w = profile_cfg["weights"]
    # band flags
    b_dx  = band_flag(prims["dx"],  bands["dx"]["guard"])
    b_rv  = band_flag(prims["rv"],  bands["rv"]["guard"])
    b_rho = band_flag(prims["rho"], bands["rho"]["guard"])

    # soft band scores
    s_dx  = gaussian_score(prims["dx"],  bands["dx"]["guard"])
    s_rv  = gaussian_score(prims["rv"],  bands["rv"]["guard"])
    s_rho = gaussian_score(prims["rho"], bands["rho"]["guard"])

    # weighted geometric mean (epsilon-safe)
    eps = 1e-6
    k = (
        max(eps, s_dx )**w["dx"] *
        max(eps, s_rv )**w["rv"] *
        max(eps, s_rho)**w["rho"]
    ) ** (1.0 / (w["dx"] + w["rv"] + w["rho"]))

    LSI = 100.0 * k
    accepted = (LSI >= cfg["accept"]["gate_100"]) and (b_dx!="RED") and (b_rv!="RED") and (b_rho!="RED")

    return {
        "delta_x": round(prims["dx"], 3),
        "void_ratio": round(prims["rv"], 3),
        "rupture_rho": round(prims["rho"], 3),
        "K_lite": round(k, 3),
        "LSI_lite_100": round(LSI, 1),
        "band_delta_x": b_dx,
        "band_r_v": b_rv,
        "band_rho_r": b_rho,
        "accepted": bool(accepted),
    }


In [ ]:

# @title Reset: clear /content/images
import shutil, os
IMG_DIR = "/content/images"
if os.path.exists(IMG_DIR):
    shutil.rmtree(IMG_DIR)
os.makedirs(IMG_DIR, exist_ok=True)
print("Reset:", IMG_DIR)


In [ ]:

# @title Option A — Classic multiple‑file uploader (preferred)
from google.colab import files
uploaded = files.upload()
upload_paths = []
for name, data in uploaded.items():
    path = f"/content/images/{name}"
    with open(path, "wb") as f:
        f.write(data)
    upload_paths.append(path)
print("Uploaded:", len(upload_paths), "files")


In [ ]:

# @title Option B — Upload a ZIP of images (fallback)
from google.colab import files
import zipfile, io, os, glob
z = files.upload()
assert len(z)==1, "Upload exactly one .zip"
name, bytes_ = next(iter(z.items()))
assert name.lower().endswith(".zip"), "This path expects a .zip file"
with zipfile.ZipFile(io.BytesIO(bytes_), 'r') as zip_ref:
    zip_ref.extractall("/content/images")
upload_paths = sorted([p for p in glob.glob("/content/images/**/*", recursive=True)
                       if os.path.splitext(p)[1].lower() in [".png",".jpg",".jpeg",".webp",".bmp",".tif",".tiff"]])
print("Extracted:", len(upload_paths), "images to /content/images")


In [ ]:

# @title Option C — Google Drive folder copy (batch)
from google.colab import drive
import shutil, os, glob
drive.mount('/content/drive')
DRIVE_FOLDER = ""  # @param {type:"string"}
assert DRIVE_FOLDER, "Set DRIVE_FOLDER to a Drive path, e.g. /content/drive/MyDrive/my_images"
ex = [".png",".jpg",".jpeg",".webp",".bmp",".tif",".tiff"]
srcs = [p for p in glob.glob(os.path.join(DRIVE_FOLDER, "**/*"), recursive=True)
        if os.path.splitext(p)[1].lower() in ex]
for p in srcs:
    shutil.copy2(p, "/content/images/")
print("Copied:", len(srcs), "images into /content/images")


In [ ]:

# @title Profile selector (widget + fallback)
try:
    import ipywidgets as widgets
    profile_mode_widget = widgets.Dropdown(
        options=["Auto","Figure_Default","MarkMaking_Expressive","Landscape"],
        value="Auto",
        description="profile_mode",
    )
    display(profile_mode_widget)
except Exception as e:
    print("Widget not available; using string fallback. Set profile_mode manually.")
profile_mode = "Auto"  # @param ["Auto","Figure_Default","MarkMaking_Expressive","Landscape"]


In [ ]:
import matplotlib.image as mpimg
import glob, os

paths = sorted(glob.glob("/content/images/*"))
n = min(12, len(paths))
if n == 0:
    print("No images in /content/images yet. Use Option A2/A3 or B above.")
else:
    cols = 4; rows = (n + cols - 1)//cols
    plt.figure(figsize=(cols*3, rows*3))
    for i,p in enumerate(paths[:n], 1):
        plt.subplot(rows, cols, i)
        plt.imshow(mpimg.imread(p))
        plt.title(os.path.basename(p)[:30], fontsize=8); plt.axis("off")
    plt.show()

import matplotlib.pyplot as plt

def show_color_row(rgb_u8, gray_u8, fg_gray_mask_u8=None):
    try:
        lab = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2LAB)
        L = lab[:, :, 0]
        fg_color_mask = color_mask_via_lab_kmeans(rgb_u8, k=3, morph_kernel=CONFIG["preprocessing"]["morph_kernel"])
        edges_L = cv2.Canny(L, 50, 100)
        edges_L_masked = cv2.bitwise_and(edges_L, edges_L, mask=fg_color_mask)

        plt.figure(figsize=(10, 3))
        plt.subplot(1, 3, 1); plt.imshow(fg_color_mask, cmap="gray"); plt.title("Color FG Mask"); plt.axis("off")
        if fg_gray_mask_u8 is not None:
            plt.subplot(1, 3, 2); plt.imshow(fg_gray_mask_u8, cmap="gray"); plt.title("Gray FG Mask"); plt.axis("off")
        else:
            plt.subplot(1, 3, 2); plt.imshow(gray_u8, cmap="gray"); plt.title("Gray"); plt.axis("off")
        plt.subplot(1, 3, 3); plt.imshow(edges_L_masked, cmap="gray"); plt.title("L-Edges ∩ Color FG"); plt.axis("off")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print("[Color row] skipped:", e)

if INCLUDE_COLOR:
    paths = sorted(glob.glob("/content/images/*"))
    if paths:
        g, r = load_image_robust(paths[0])
        show_color_row(r, g)

In [ ]:
# @title Run scoring (per-image loop)
# === Run scoring (per-image loop) ===
from glob import glob
import os, numpy as np, pandas as pd

# ---- 0) Gather inputs (whatever Option A/B/C uploaded) ----
exts = (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff")
upload_paths = sorted([p for p in glob("/content/images/*")
                       if os.path.splitext(p)[1].lower() in exts])
print("Found", len(upload_paths), "images")

# ---- 1) Read chosen mode from the dropdown (or fallback string) ----
try:
    chosen_mode = profile_mode_widget.value
except NameError:
    chosen_mode = profile_mode
print("Using profile_mode:", chosen_mode)

rows = []
def _reason_for_fail(r):
    if any(r.get(k) == "RED" for k in ("band_delta_x","band_r_v","band_rho_r")):
        return "band_red"
    if r.get("LSI_lite_100", 0) < CONFIG["accept"]["gate_100"]:
        return "lsi_gate"
    return ""

for path in upload_paths:
    # ---- load image once (EXIF-aware, robust); ignore RGB in this loop ----
    gray, rgb = load_image_robust(path)

    # ---- choose profile ----
    if chosen_mode == "Auto":
        # quick, ROI-free probes just to decide a profile
        quick_dx = centroid_delta_x(gray, CONFIG, dx_roi=None)
        quick_rv = void_ratio(gray, CONFIG)
        pf = auto_profile(quick_dx, quick_rv)
    else:
        pf = chosen_mode

    # mask settings for this profile
    rho_mask_cfg = PROFILES[pf]["rho_mask"]

    # ---- optional ROI for Δx (Landscape only, if configured) ----
    dx_roi = None
    if pf == "Landscape" and PROFILES[pf].get("dx_roi") == "auto":
        dx_roi = auto_dx_roi(gray) or None     # returns {"type":"top_frac","top":0.60} or None

    # ---- compute ALL primitives ONCE (threads dx_roi into Δx) ----
    prims = measure_primitives(gray, CONFIG, rho_mask_cfg, dx_roi=dx_roi)

    # --- Color telemetry with optional external subject mask (no impact on acceptance) ---
    rv_color = dx_color = None
    mask_mode, color_status = "gray", "fallback_gray"

    # 1) Provide an external mask if you have one; otherwise leave as None.
    #    Requirement: uint8 HxW, values 0 or 255, SAME size as rgb.
    external_mask_u8 = None
    # Example (optional): auto-load a sidecar mask named "<image>_mask.png"
    # import os, cv2
    # sidecar = os.path.splitext(path)[0] + "_mask.png"
    # if os.path.exists(sidecar):
    #     m = cv2.imread(sidecar, cv2.IMREAD_GRAYSCALE)
    #     if m is not None and m.shape[:2] == rgb.shape[:2]:
    #         external_mask_u8 = (m > 0).astype("uint8") * 255

    if INCLUDE_COLOR and rgb is not None:
        if isinstance(external_mask_u8, np.ndarray):
            lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
            L_full = lab[:, :, 0]
            if isinstance(dx_roi, dict) and dx_roi.get("type") == "top_frac":
                L_roi  = apply_roi_to_L(L_full, dx_roi)
                m_roi  = apply_roi_to_L(external_mask_u8, dx_roi)
            else:
                L_roi, m_roi = L_full, external_mask_u8
            rv_color = compute_rv_from_mask(m_roi)
            dx_color = compute_dx_color(L_roi, m_roi)
            mask_mode, color_status = "external", "ok"
        else:
            # Use color-derived mask, aligned to the same ROI as Δx
            lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
            L_full = lab[:, :, 0]
            color_fg = color_mask_via_lab_kmeans(
                rgb, k=3, morph_kernel=CONFIG["preprocessing"]["morph_kernel"]
            )

            if isinstance(dx_roi, dict) and dx_roi.get("type") == "top_frac":
                L_roi    = apply_roi_to_L(L_full, dx_roi)
                fg_roi   = apply_roi_to_L(color_fg, dx_roi)
            else:
                L_roi, fg_roi = L_full, color_fg

            rv_color = compute_rv_from_mask(fg_roi)
            dx_color = compute_dx_color(L_roi, fg_roi)
            mask_mode = "color"
            color_status = "ok"

    # ---- score with the selected profile ----
    row = score_with_profile(prims, PROFILES[pf], CONFIG)
    row.update({"frame": os.path.basename(path), "profile": pf})

    # --- COLOR DELTAS + AUDIT (non-gating) ---
    import math
    delta_rv = (rv_color - prims["rv"]) if (rv_color is not None and not math.isnan(prims["rv"])) else float("nan")
    delta_dx = (dx_color - prims["dx"]) if (dx_color is not None and not math.isnan(prims["dx"])) else float("nan")

    color_audit_badge = ""
    if color_status == "ok":
        if (abs(delta_rv) >= 0.15) or (abs(delta_dx) >= 0.10) or near_guard_rv(prims["rv"], pf, tol=0.02):
            color_audit_badge = (
               f"Color audit: d_rv={delta_rv:+.2f}, ddx={delta_dx:+.2f} — check subject mask / glare"
            )

    # --- TONAL TELEMETRY + AUDIT (non-gating) ---
    tonal_status = "ok"; tonal_audit_badge = ""
    S_L = eta_L = beta_L = float("nan")
    try:
        lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
        L = apply_roi_to_L(lab[:, :, 0], dx_roi)  # same ROI as Δx
        if L.size == 0 or L.var() < 1e-6:
            tonal_status = "flat_or_empty"
        else:
            S_L, eta_L, beta_L = tonal_stats_L(L)
            if (abs(delta_dx) >= 0.10) or (S_L <= 0.12) or (S_L >= 0.60) or (eta_L <= 0.35 and prims["rho"] >= 0.45):
                tonal_audit_badge = (
                    f"Tonal audit: span={S_L:.2f}, sep={eta_L:.2f}; ddx={delta_dx:+.2f} — adjust light/crop/mask"
            )
    except Exception:
        tonal_status = "fail_convert"

    # --- Combine badges to avoid spam ---
    if color_audit_badge and tonal_audit_badge:
        combined = "Color/Tonal audit: " + "; ".join([
            color_audit_badge.replace("Color audit: ", ""),
            tonal_audit_badge.replace("Tonal audit: ", "")
        ])
        color_audit_badge = tonal_audit_badge = combined

    # --- Attach telemetry to the row ---
    row.update({
        # color telemetry
        "rv_color_mask": rv_color,
        "dx_color_L": dx_color,
        "delta_rv": delta_rv,
        "delta_dx": delta_dx,
        "mask_mode": mask_mode,
        "color_space": COLOR_SPACE,
        "color_status": color_status,
        "color_audit_badge": color_audit_badge,
        "S_L": S_L, "eta_L": eta_L, "beta_L": beta_L,
        "tonal_status": tonal_status, "tonal_audit_badge": tonal_audit_badge,
    })

    # --- reason_for_fail (helper defined once above the loop) ---
    row["reason_for_fail"] = _reason_for_fail(row)

    rows.append(row)


# ----------------------------------------------------------------------------------
# Post-processing helper: band centers, distances, normalized offsets, and priority
# ----------------------------------------------------------------------------------

import numpy as np, pandas as pd

def add_band_distance_columns(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()

    c_dx  = out["profile"].apply(lambda pf: band_center_and_span(pf,"dx")[0]).astype(float)
    c_rv  = out["profile"].apply(lambda pf: band_center_and_span(pf,"rv")[0]).astype(float)
    c_rho = out["profile"].apply(lambda pf: band_center_and_span(pf,"rho")[0]).astype(float)
    s_dx  = out["profile"].apply(lambda pf: band_center_and_span(pf,"dx")[1]).astype(float)
    s_rv  = out["profile"].apply(lambda pf: band_center_and_span(pf,"rv")[1]).astype(float)
    s_rho = out["profile"].apply(lambda pf: band_center_and_span(pf,"rho")[1]).astype(float)

    def _emit(col, center, span, key):
        lo, hi = center - span, center + span
        vals   = out[col].astype(float)
        inside = (vals >= lo) & (vals <= hi)
        to_c   = (vals - center).abs()
        to_c_n = np.clip(to_c / np.maximum(span, 1e-9), 0.0, 1.0)
        out[f"{key}_inside_band"]    = inside
        out[f"{key}_to_center"]      = to_c
        out[f"{key}_to_center_norm"] = to_c_n
        out[f"{key}_direction"]      = np.where(vals < center, "increase", "decrease")

    _emit("delta_x",     c_dx,  s_dx,  "dx")
    _emit("void_ratio",  c_rv,  s_rv,  "rv")
    _emit("rupture_rho", c_rho, s_rho, "rho")

    # weighted farthest-from-center
    def _row_priority(row):
        w = PROFILES[row["profile"]]["weights"]
        scores = {
            "dx":  w["dx"]  * row["dx_to_center_norm"],
            "rv":  w["rv"]  * row["rv_to_center_norm"],
            "rho": w["rho"] * row["rho_to_center_norm"],
        }
        knob = max(scores, key=scores.get)
        return pd.Series({"priority_knob": knob, "priority_score": float(scores[knob])})
    out[["priority_knob","priority_score"]] = out.apply(_row_priority, axis=1)

    return out

def polish_df(df_in: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "delta_x","void_ratio","rupture_rho","K_lite","LSI_lite_100",
        "band_delta_x","band_r_v","band_rho_r","accepted","frame","profile",
        "dx_inside_band","dx_to_center","dx_to_center_norm","dx_direction",
        "rv_inside_band","rv_to_center","rv_to_center_norm","rv_direction",
        "rho_inside_band","rho_to_center","rho_to_center_norm","rho_direction",
        "priority_knob","priority_score",
        "rv_color_mask","dx_color_L","delta_rv","delta_dx",
        "mask_mode","color_space","color_status","color_audit_badge",
        "S_L","eta_L","beta_L","tonal_status","tonal_audit_badge",
        "reason_for_fail",
    ]
    return df_in[[c for c in cols if c in df_in.columns]]

# --- Build the base table (call AFTER defs) ---
base = pd.DataFrame(rows)               # no rounding yet
df = add_band_distance_columns(base)
df = polish_df(df).round(3)
display(df)


In [ ]:
# 2) Disable truncation for the session
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)
df.head()
display(df)

In [ ]:

# @title Plots
import matplotlib.pyplot as plt
if 'df' in globals() and not df.empty:
    xs = list(range(1, len(df)+1))
    plt.figure(figsize=(6,4))
    plt.plot(xs, df["delta_x"], label="Δx")
    plt.plot(xs, df["void_ratio"], label="r_v")
    plt.plot(xs, df["rupture_rho"], label="ρ_r")
    plt.xlabel("iteration"); plt.ylabel("value (0–1)"); plt.title("Primitives over iterations")
    plt.legend(); plt.show()

    plt.figure(figsize=(6,4))
    plt.plot(xs, df["LSI_lite_100"])
    plt.xlabel("iteration"); plt.ylabel("LSI_lite_100 (0–100)"); plt.title("LSI_lite_100 over iterations")
    plt.show()

# Audit rate (Color or Tonal)
    mask_color = df["color_audit_badge"].astype(str).str.len() > 0
    mask_tonal = df["tonal_audit_badge"].astype(str).str.len() > 0 if "tonal_audit_badge" in df.columns else False
    audit_rate = (mask_color | mask_tonal).mean() if len(df) else 0.0
    print(f"Audit rate: {audit_rate*100:.1f}% (Color or Tonal)")
else:
    print("No dataframe to plot. Run scoring first.")


In [ ]:

# @title Export (CSV + simple HTML)
import pandas as pd, os
CSV_PATH = "/content/LSI_lite_results.csv"
HTML_PATH = "/content/LSI_lite_report.html"
if 'df' in globals() and not df.empty:
    df.to_csv(CSV_PATH, index=False)
    html = "<h2>LSI_lite results</h2>" + df.to_html(index=False)
    with open(HTML_PATH, "w") as f:
        f.write(html)
    print("Saved:", CSV_PATH, "and", HTML_PATH)
else:
    print("Nothing to export; run scoring first.")


### Notes & tips
- If the classic uploader throws a browser **RangeError** on big files, use **ZIP** or **Drive**.
- Keep `PIN_DEPS=False` unless you *need* exact versions; pinning can be slow in Colab.
- `Auto` is for smoke tests. For grading, set a fixed profile (esp. MarkMaking vs Landscape).
- If you see false **RED ρᵣ** on charcoal: widen the `rho.guard` upper bound OR increase `halo_frac`.
- Acceptance rule: `LSI_lite_100 ≥ gate` **and** all bands `OK`.
